In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_df = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/kidney_stone/train.csv')

# Display the first few rows of the dataframe
train_df.head()

# Check the data types of each column
train_df.dtypes

# Summary statistics for numerical columns
train_df.describe()

# Check for missing values
train_df.isnull().sum()

# Distribution of the target variable
sns.countplot(x='target', data=train_df)
plt.title('Distribution of Target Variable')
plt.show()

# Correlation matrix for numerical features
corr_matrix = train_df.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numerical Features')
plt.show()

# Distribution of numerical features
numerical_features = train_df.select_dtypes(include=[np.number]).columns.tolist()
for feature in numerical_features:
    plt.figure(figsize=(8, 4))
    sns.histplot(train_df[feature], kde=True)
    plt.title(f'Distribution of {feature}')
    plt.show()


In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-08-30 20:14:33.998 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': [], 'Numeric': ['id', 'gravity', 'ph', 'osmo', 'cond', 'urea', 'calc', 'target'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, StandardScale

# Copy the DataFrame to avoid modifying the original data
train_df_copy = train_df.copy()

# Handling missing values
# Since all features are numeric, we can use the mean strategy to fill missing values
fill_missing = FillMissingValue(features=numerical_features, strategy='mean')
train_df_processed = fill_missing.fit_transform(train_df_copy)

# Scaling numerical features
# We will use standard scaling for numerical features
scaler = StandardScale(features=numerical_features)
train_df_processed = scaler.fit_transform(train_df_processed)

# Display the first few rows of the processed DataFrame
train_df_processed.head()


,id,gravity,ph,osmo,cond,urea,calc,target
0,-0.158080,-0.932744,-0.271945,-0.866679,-0.563507,-0.646711,-0.844550,-0.888363
1,0.200248,-0.173427,-0.364694,0.196437,0.387993,-0.087629,-0.190738,-0.888363
2,-1.753491,1.041479,1.474843,1.259554,0.910648,0.844175,-0.450993,1.125666
3,-1.412227,-1.540197,0.052680,0.524560,-0.509901,1.015627,0.929630,1.125666
4,0.294096,1.952659,-1.091234,0.192062,0.267380,0.613087,2.735547,1.125666


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df_processed)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'gravity', 'ph', 'osmo', 'cond', 'urea', 'calc', 'target'], 'Datetime': [], 'Others': []}


In [5]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Split the data into features and target
X = train_df_processed.drop(columns=['id', 'target'])
y = train_df_processed['target']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize the XGBoost classifier
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# Define the parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Perform grid search to find the best parameters
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best model
best_model = grid_search.best_estimator_

# Predict probabilities on the test set
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Calculate the AUC-ROC score
auc_roc = roc_auc_score(y_test, y_pred_proba)
print(f'AUC-ROC Score: {auc_roc}')


ValueError: 
All the 540 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
540 fits failed with the following error:
Traceback (most recent call last):
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\sklearn\model_selection\_validation.py", line 729, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\xgboost\core.py", line 726, in inner_f
    return func(**kwargs)
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter\venv\lib\site-packages\xgboost\sklearn.py", line 1491, in fit
    raise ValueError(
ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got [-0.88836321  1.12566571]
